# BM25 で日本語のキーワード検索をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_similarity_search/bm25_ja.ipynb)

NLP クックブック 12-04 ／ 文埋め込み・類似度・検索

単語の一致に基づく古典的な検索 (BM25) を日本語文書に対して行いたい。固有名詞や型番の検索には今でも強い。

In [1]:
!pip install -q rank_bm25 fugashi unidic-lite

In [2]:
from fugashi import Tagger
from rank_bm25 import BM25Okapi

tagger = Tagger()

def tokenize(text):
    return [w.feature.orthBase or w.surface for w in tagger(text.lower())
            if w.feature.pos1 not in ("助詞", "助動詞", "補助記号", "空白")]

docs = [
    "富士山は標高 3776 メートルで、日本で最も高い山である。",
    "東京スカイツリーは高さ 634 メートルの電波塔である。",
    "琵琶湖は滋賀県にある日本最大の湖である。",
    "信濃川は全長 367 キロメートルで、日本で最も長い川である。",
    "形態素解析とは、文を単語に分割して品詞を付与する処理である。",
    "BERT は Transformer のエンコーダを用いた事前学習モデルである。",
]
bm25 = BM25Okapi([tokenize(d) for d in docs])

for q in ["日本で一番高い山", "BERT とは何ですか", "日本一大きい湖はどこ？"]:
    scores = bm25.get_scores(tokenize(q))
    print(q, tokenize(q))
    for i in scores.argsort()[::-1][:2]:
        print(f"  {scores[i]:.2f}  {docs[i]}")

日本で一番高い山 ['日本', '一番', '高い', '山']
  2.41  富士山は標高 3776 メートルで、日本で最も高い山である。
  0.58  東京スカイツリーは高さ 634 メートルの電波塔である。
BERT とは何ですか ['bert', '何']
  1.41  BERT は Transformer のエンコーダを用いた事前学習モデルである。
  0.00  形態素解析とは、文を単語に分割して品詞を付与する処理である。
日本一大きい湖はどこ？ ['日本', '一', '大きい', '湖', 'どこ']
  1.90  琵琶湖は滋賀県にある日本最大の湖である。
  0.00  BERT は Transformer のエンコーダを用いた事前学習モデルである。


- 3 つ目のクエリのように語が一致しない言い換え (大きい ↔ 最大) には弱い。埋め込み検索と組み合わせると補い合える (次項)。
- 大規模なら `bm25s` (高速な実装) や Elasticsearch / OpenSearch、Pyserini を使う。